# Cell 1 — Import libraries

In [2]:
# Import pandas for reading CSV files and working with table data
import pandas as pd

# Import NumPy for numerical operations
import numpy as np

# Import train_test_split to divide data into training and testing sets
from sklearn.model_selection import train_test_split

# Import LabelEncoder to convert text labels into numeric values
from sklearn.preprocessing import LabelEncoder

# Import RandomForestClassifier as the machine learning model
from sklearn.ensemble import RandomForestClassifier

# Import accuracy_score to measure prediction accuracy
from sklearn.metrics import accuracy_score

# Import confusion_matrix to see correct and incorrect predictions by class
from sklearn.metrics import confusion_matrix

# Import classification_report to show precision, recall, and F1-score
from sklearn.metrics import classification_report

# Import pickle so we can save the trained model and encoders into a file
import pickle

# Cell 2 — Load dataset

In [3]:
# Read the CSV file from the data folder into a pandas DataFrame named df
df = pd.read_csv("bmw_global_sales_2018_2025.csv")

# Show the first 5 rows of the dataset so we can preview the data
df.head()

,Year,Month,Region,Model,Units_Sold,Avg_Price_EUR,Revenue_EUR,BEV_Share,Premium_Share,GDP_Growth,Fuel_Price_Index
0,2018,1,Europe,3 Series,7822,47482,371404204,0.011,19.12,3.5,1.0
1,2018,1,Europe,5 Series,10280,61685,634121800,0.019,19.12,3.5,1.0
2,2018,1,Europe,X3,3105,58433,181434465,0.022,19.12,3.5,1.0
3,2018,1,Europe,X5,7420,67955,504226100,0.021,19.12,3.5,1.0
4,2018,1,Europe,X7,8474,92300,782150200,0.035,19.12,3.5,1.0


# Cell 3 — Check dataset shape

In [4]:
# Show the number of rows and columns in the dataset
df.shape

(3072, 11)

# Cell 4 — Show column names

In [5]:
# Show all column names in the dataset
df.columns

Index(['Year', 'Month', 'Region', 'Model', 'Units_Sold', 'Avg_Price_EUR',
       'Revenue_EUR', 'BEV_Share', 'Premium_Share', 'GDP_Growth',
       'Fuel_Price_Index'],
      dtype='object')

# Cell 5 — Check dataset info

In [6]:
# Show detailed information such as column names, data types, and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3072 entries, 0 to 3071
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Year              3072 non-null   int64  
 1   Month             3072 non-null   int64  
 2   Region            3072 non-null   object 
 3   Model             3072 non-null   object 
 4   Units_Sold        3072 non-null   int64  
 5   Avg_Price_EUR     3072 non-null   int64  
 6   Revenue_EUR       3072 non-null   int64  
 7   BEV_Share         3072 non-null   float64
 8   Premium_Share     3072 non-null   float64
 9   GDP_Growth        3072 non-null   float64
 10  Fuel_Price_Index  3072 non-null   float64
dtypes: float64(4), int64(5), object(2)
memory usage: 264.1+ KB


# Cell 6 — Summary statistics

In [7]:
# Show summary statistics for numerical columns such as mean, min, max, and quartiles
df.describe()

,Year,Month,Units_Sold,Avg_Price_EUR,Revenue_EUR,BEV_Share,Premium_Share,GDP_Growth,Fuel_Price_Index
count,3072.000000,3072.000000,3072.000000,3072.000000,3.072000e+03,3072.000000,3072.000000,3072.000000,3072.000000
mean,2021.500000,6.500000,7980.288086,63854.561523,5.113995e+08,0.107572,14.660234,3.057526,1.176224
std,2.291661,3.452615,3174.917444,14655.891299,2.431185e+08,0.058099,5.334604,1.018103,0.118123
min,2018.000000,1.000000,2379.000000,40011.000000,1.045314e+08,-0.015000,5.090000,0.100000,0.920000
25%,2019.750000,3.750000,5225.500000,54500.250000,3.125504e+08,0.057000,12.275000,2.370000,1.080000
50%,2021.500000,6.500000,7985.500000,63493.000000,4.808690e+08,0.108000,16.260000,3.060000,1.180000
75%,2023.250000,9.250000,10528.250000,71489.500000,6.709141e+08,0.157250,18.932500,3.852500,1.270000
max,2025.000000,12.000000,15914.000000,93994.000000,1.433482e+09,0.223000,20.970000,5.820000,1.410000


# Cell 7 — Create target column for classification

In [8]:
# Create a new column named High_Sales based on Units_Sold compared with the median
df["High_Sales"] = (df["Units_Sold"] >= df["Units_Sold"].median()).astype(int)

# Show the first 5 rows so we can confirm the new High_Sales column was created
df.head()

,Year,Month,Region,Model,Units_Sold,Avg_Price_EUR,Revenue_EUR,BEV_Share,Premium_Share,GDP_Growth,Fuel_Price_Index,High_Sales
0,2018,1,Europe,3 Series,7822,47482,371404204,0.011,19.12,3.5,1.0,0
1,2018,1,Europe,5 Series,10280,61685,634121800,0.019,19.12,3.5,1.0,1
2,2018,1,Europe,X3,3105,58433,181434465,0.022,19.12,3.5,1.0,0
3,2018,1,Europe,X5,7420,67955,504226100,0.021,19.12,3.5,1.0,0
4,2018,1,Europe,X7,8474,92300,782150200,0.035,19.12,3.5,1.0,1


# Cell 8 — Check target balance

In [9]:
# Count how many rows belong to each class in High_Sales
df["High_Sales"].value_counts()

High_Sales
0    1536
1    1536
Name: count, dtype: int64

# Cell 9 — Make a fresh copy for modeling

In [10]:
# Make a fresh copy of df into df_model so we can safely transform data for modeling
df_model = df.copy()

# Show the first 5 rows of df_model
df_model.head()

,Year,Month,Region,Model,Units_Sold,Avg_Price_EUR,Revenue_EUR,BEV_Share,Premium_Share,GDP_Growth,Fuel_Price_Index,High_Sales
0,2018,1,Europe,3 Series,7822,47482,371404204,0.011,19.12,3.5,1.0,0
1,2018,1,Europe,5 Series,10280,61685,634121800,0.019,19.12,3.5,1.0,1
2,2018,1,Europe,X3,3105,58433,181434465,0.022,19.12,3.5,1.0,0
3,2018,1,Europe,X5,7420,67955,504226100,0.021,19.12,3.5,1.0,0
4,2018,1,Europe,X7,8474,92300,782150200,0.035,19.12,3.5,1.0,1


# Cell 10 — Encode Region column

In [11]:
# Create a LabelEncoder object for the Region column
region_encoder = LabelEncoder()

# Convert Region text values such as Europe, China, USA into numeric labels
df_model["Region"] = region_encoder.fit_transform(df_model["Region"])

# Show the first 5 rows after encoding Region
df_model.head()


,Year,Month,Region,Model,Units_Sold,Avg_Price_EUR,Revenue_EUR,BEV_Share,Premium_Share,GDP_Growth,Fuel_Price_Index,High_Sales
0,2018,1,1,3 Series,7822,47482,371404204,0.011,19.12,3.5,1.0,0
1,2018,1,1,5 Series,10280,61685,634121800,0.019,19.12,3.5,1.0,1
2,2018,1,1,X3,3105,58433,181434465,0.022,19.12,3.5,1.0,0
3,2018,1,1,X5,7420,67955,504226100,0.021,19.12,3.5,1.0,0
4,2018,1,1,X7,8474,92300,782150200,0.035,19.12,3.5,1.0,1


# Cell 11 — Encode Model column

In [12]:
# Create a LabelEncoder object for the Model column
model_encoder = LabelEncoder()

# Convert Model text values such as 3 Series, X5, MINI into numeric labels
df_model["Model"] = model_encoder.fit_transform(df_model["Model"])

# Show the first 5 rows after encoding Model
df_model.head()


,Year,Month,Region,Model,Units_Sold,Avg_Price_EUR,Revenue_EUR,BEV_Share,Premium_Share,GDP_Growth,Fuel_Price_Index,High_Sales
0,2018,1,1,0,7822,47482,371404204,0.011,19.12,3.5,1.0,0
1,2018,1,1,1,10280,61685,634121800,0.019,19.12,3.5,1.0,1
2,2018,1,1,3,3105,58433,181434465,0.022,19.12,3.5,1.0,0
3,2018,1,1,4,7420,67955,504226100,0.021,19.12,3.5,1.0,0
4,2018,1,1,5,8474,92300,782150200,0.035,19.12,3.5,1.0,1


# Cell 12 — Build feature matrix X and target y using Year, Region, and Model

In [13]:
# Select the input features Model, Year, and Region
X = df_model[["Year", "Region", "Model"]].copy()

# Select the target column
y = df_model["High_Sales"].copy()

# Show the first 5 rows of X
X.head()


,Year,Region,Model
0,2018,1,0
1,2018,1,1
2,2018,1,3
3,2018,1,4
4,2018,1,5


# Cell 13 — Check target values

In [14]:
# Show the first 5 values of y so we can verify the target labels
y.head()


0    0
1    1
2    0
3    0
4    1
Name: High_Sales, dtype: int64

# Cell 14 — Check data types in X before training

In [15]:
# Print the data type of each feature column in X
print(X.dtypes)

# Print any remaining object or text columns in X to make sure all features are numeric
print(X.select_dtypes(include=["object"]).columns)


Year      int64
Region    int64
Model     int64
dtype: object
Index([], dtype='object')


# Cell 15 — Extra safety conversion for any leftover text columns

In [16]:
# Loop through every column in X that still has object or text data type
for col in X.select_dtypes(include=["object"]).columns:
    # Convert that text column into category codes so the model can use numbers instead of strings
    X[col] = X[col].astype("category").cat.codes

# Print the data types again after conversion
print(X.dtypes)


Year      int64
Region    int64
Model     int64
dtype: object


# Cell 16 — Split into training and testing sets

In [17]:
# Split X and y into training and testing sets using 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Print the shape of X_train so we know the size of training features
print("X_train shape:", X_train.shape)

# Print the shape of X_test so we know the size of testing features
print("X_test shape:", X_test.shape)


X_train shape: (2457, 3)
X_test shape: (615, 3)


# Cell 17 — Build and train model

In [18]:
# Create a Random Forest classifier model with a fixed random state for reproducibility
model = RandomForestClassifier(random_state=42)

# Train the model using the training feature data and training target labels
model.fit(X_train, y_train)

# Confirm that the model has been trained
print("Model trained successfully")


Model trained successfully


# Cell 18 — Make predictions

In [19]:
# Use the trained model to predict target labels for the testing feature data
y_pred = model.predict(X_test)

# Show the first 10 predicted values
y_pred[:10]


array([1, 1, 0, 0, 1, 1, 0, 1, 0, 0])

# Cell 19 — Check accuracy

In [20]:
# Calculate the prediction accuracy by comparing true labels and predicted labels
accuracy = accuracy_score(y_test, y_pred)

# Print the accuracy result
print("Accuracy:", accuracy)


Accuracy: 0.4894308943089431


# Cell 20 — Confusion matrix

In [21]:
# Generate the confusion matrix to compare actual labels against predicted labels
cm = confusion_matrix(y_test, y_pred)

# Print the confusion matrix
print(cm)


[[149 131]
 [183 152]]


# Cell 21 — Classification report

In [22]:
# Generate a classification report with precision, recall, and F1-score
report = classification_report(y_test, y_pred)

# Print the classification report
print(report)


              precision    recall  f1-score   support

           0       0.45      0.53      0.49       280
           1       0.54      0.45      0.49       335

    accuracy                           0.49       615
   macro avg       0.49      0.49      0.49       615
weighted avg       0.50      0.49      0.49       615



# Cell 22 — Save model, encoders, and feature list

In [23]:
# Create a dictionary that stores everything needed later for prediction in Flask
saved_objects = {
    "model": model,
    "region_encoder": region_encoder,
    "model_encoder": model_encoder,
    "feature_columns": X.columns.tolist(),
}

# Save the dictionary into a file named model.pkl
with open("model.pkl", "wb") as f:
    pickle.dump(saved_objects, f)

# Print a success message
print("Saved model.pkl successfully")


Saved model.pkl successfully


# Cell 23 — Test loading the saved file back

In [24]:
# Open the saved model.pkl file in read-binary mode
with open("model.pkl", "rb") as f:
    # Load the saved dictionary back into memory
    loaded_objects = pickle.load(f)

# Show the keys inside the loaded dictionary
print(loaded_objects.keys())


dict_keys(['model', 'region_encoder', 'model_encoder', 'feature_columns'])


# Cell 24 — Optional sample prediction with Year, Region, and Model

In [25]:
# Create one sample input row using Year, Region, and Model
sample_input = pd.DataFrame(
    [
        {
            "Year": 2025,
            "Region": region_encoder.transform(["Europe"])[0],
            "Model": model_encoder.transform(["i4"])[0]
        }
    ]
)

# Show the sample input
sample_input


,Year,Region,Model
0,2025,1,6


# Cell 25 — Predict from the sample input and convert class to label

In [26]:
# Use the trained model to predict the class for the sample input
sample_prediction = model.predict(sample_input)

# Convert the numeric prediction into a readable label
prediction_label = "High Sale" if sample_prediction[0] == 1 else "Low Sale"

# Print the result with the selected Year, Region, and Model
print("Model:", "i4")
print("Year:", 2025)
print("Region:", "Europe")
print("Predicted sales class:", prediction_label)


Model: i4
Year: 2025
Region: Europe
Predicted sales class: Low Sale
